# Pelvis-SigLIP — try it on your own scan

The only thing you need to edit is `NII_PATH` below — everything else runs as-is.

This notebook does three things, each explained as it goes:
1. **Loads a `.nii`/`.nii.gz` volume** and extracts a 2D slice, the same kind of input the models expect.
2. **Zero-shot classification** — the naive baseline from the paper: embed the image and a set of text prompts with the *base*, non-fine-tuned SigLIP model, and pick the closest text prompt by cosine similarity. The paper shows this **fails** (scores below a majority-class baseline) — it's included here so you can see *why* fine-tuning was necessary, not just take it on faith.
3. **The two released Pelvis-SigLIP checkpoints** — MedSigLIP (sequence type) and SigLIP-2 (view orientation) — on the same slice.

Nothing beyond this repo and a local NIfTI file is required. Model weights download automatically from the Hugging Face Hub the first time you run it.

## Setup

```bash

pip install -r ../requirements.txt

```

`google/medsiglip-448` is gated on Hugging Face — request access on the model page, then run `huggingface-cli login` once before executing this notebook.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.data import preprocess_image_array
from src.models import MedSigLIPModel, SigLIP2Model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 1. Point this at your scan

Edit `NII_PATH` to a local `.nii` or `.nii.gz` file. Everything else in this section is fixed logic — you shouldn't need to touch it for a typical pelvic MRI volume.

> **Note on preprocessing:** the exact slice-extraction script used to build the paper's training thumbnails wasn't part of this release. What's below is a standard, well-documented approach — central axial slice, percentile intensity windowing — that produces the same *kind* of image (a single normalised 2D slice) the models expect. If your volumes have unusual orientation codes or unusually dark/bright windowing, sanity-check the displayed slice below before trusting predictions.

In [ ]:
NII_PATH = "path/to/your/scan.nii.gz"   # <-- EDIT THIS
SLICE_AXIS = 2   # 0=sagittal, 1=coronal, 2=axial for a typical RAS-oriented volume; check the preview below


In [ ]:
import nibabel as nib

def load_central_slice(nii_path: str, axis: int = 2) -> np.ndarray:
    """Load a NIfTI volume and return the central slice along `axis` as a
    uint8 grayscale array, percentile-windowed to full contrast."""
    img = nib.load(nii_path)
    vol = img.get_fdata()
    if vol.ndim == 4:  # some series stack multiple volumes/echoes — take the first
        vol = vol[..., 0]

    idx = vol.shape[axis] // 2
    sl = np.take(vol, idx, axis=axis)

    # percentile windowing (robust to outlier bright/dark voxels)
    lo, hi = np.percentile(sl, [1, 99])
    sl = np.clip((sl - lo) / max(hi - lo, 1e-6), 0, 1)
    sl = (sl * 255).astype(np.uint8)

    # orient roughly "head up" for display; harmless if already correct
    sl = np.rot90(sl)
    return sl

slice_arr = load_central_slice(NII_PATH, axis=SLICE_AXIS)
print("Slice shape:", slice_arr.shape)

plt.figure(figsize=(5, 5))
plt.imshow(slice_arr, cmap="gray")
plt.title("Extracted central slice — check this looks like a sensible pelvic MRI slice")
plt.axis("off")
plt.show()

## 2. Zero-shot classification (the baseline that fails)

Same text-prompt ensembles used for the paper's zero-shot evaluation — three phrasings per class, averaged into one text embedding, compared to the image embedding by cosine similarity with the **base** SigLIP-SO400M-384 model (no fine-tuning). The paper's finding: every model scores *below* a majority-class baseline here. That result is the entire motivation for sections 3–4 below.

In [ ]:
SEQ_PROMPTS = {
    "T1":    ["T1-weighted MRI of the female pelvis", "T1 MRI pelvic scan", "T1w pelvic MRI"],
    "T1FS":  ["T1-weighted fat-saturated MRI of the female pelvis", "T1 fat sat pelvic MRI", "T1 FS pelvic scan with fat suppression"],
    "T2":    ["T2-weighted MRI of the female pelvis", "T2w pelvic MRI", "T2 MRI pelvic scan"],
    "T2FS":  ["T2-weighted fat-saturated MRI of the female pelvis", "T2 fat sat pelvic MRI", "T2 FS pelvic scan with fat suppression"],
    "DCE":   ["dynamic contrast-enhanced MRI of the female pelvis", "DCE MRI pelvic scan after contrast injection", "post-contrast T1 pelvic MRI"],
    "DWI":   ["diffusion-weighted imaging of the female pelvis", "DWI pelvic MRI", "diffusion MRI pelvic scan"],
    "ADC":   ["apparent diffusion coefficient map of the female pelvis", "ADC map pelvic MRI", "diffusion ADC pelvic scan"],
    "other": ["other MRI sequence of the female pelvis", "unspecified pelvic MRI sequence", "mixed or unknown pelvic MRI"],
}

ORIENT_PROMPTS = {
    "axial":    ["axial MRI slice of the female pelvis", "transverse pelvic MRI", "axial pelvic scan"],
    "sagittal": ["sagittal MRI slice of the female pelvis", "sagittal pelvic MRI", "midline sagittal pelvic scan"],
    "coronal":  ["coronal MRI slice of the female pelvis", "frontal pelvic MRI", "coronal pelvic scan"],
    "oblique":  ["oblique MRI slice of the female pelvis", "oblique pelvic MRI", "angled pelvic scan"],
}

In [ ]:
import open_clip

print("Loading base SigLIP-SO400M-384 (not fine-tuned)...")
zs_model, _, _ = open_clip.create_model_and_transforms("ViT-SO400M-14-SigLIP-384", pretrained="webli")
zs_model = zs_model.eval().to(device)
zs_tokenizer = open_clip.get_tokenizer("ViT-SO400M-14-SigLIP-384")

zs_preprocess = lambda arr: preprocess_image_array(arr, img_size=384, mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]).to(device)

@torch.no_grad()
def zeroshot_predict(slice_arr: np.ndarray, prompts_dict: dict):
    x = zs_preprocess(slice_arr)
    img_feat = zs_model.encode_image(x)
    img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)

    classes, sims = [], []
    for cls, prompts in prompts_dict.items():
        tokens = zs_tokenizer(prompts).to(device)
        text_feat = zs_model.encode_text(tokens)
        text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)
        text_feat = text_feat.mean(0, keepdim=True)
        text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)
        sims.append((img_feat @ text_feat.T).item())
        classes.append(cls)

    order = np.argsort(sims)[::-1]
    for i in order:
        marker = " <-- top" if i == order[0] else ""
        print(f"  {classes[i]:>8s}  cos_sim={sims[i]:+.4f}{marker}")
    return classes[order[0]]

print("\n[Zero-shot] Sequence type:")
zs_seq = zeroshot_predict(slice_arr, SEQ_PROMPTS)
print("\n[Zero-shot] View orientation:")
zs_orient = zeroshot_predict(slice_arr, ORIENT_PROMPTS)

del zs_model
torch.cuda.empty_cache()

Take these two predictions with a grain of salt — per the paper, zero-shot sequence accuracy for SigLIP-family models is **14.5–29.4%** (an 8-class problem where always guessing "T2" gets 36.2%). That's the whole motivation for sections 3–4.

## 3. Load the fine-tuned Pelvis-SigLIP checkpoints

Downloads both checkpoints from the Hugging Face Hub the first time you run this cell (cached locally after that — subsequent runs are fast).

In [ ]:
import json
import torch.nn as nn

CHECKPOINTS_DIR = ROOT / "checkpoints"  # only used if you have local checkpoints from your own training run
HF_REPO_IDS = {
    "MedSigLIP": "MaximilianLindholz/pelvis-siglip-medsiglip-sequence",
    "SigLIP-2":  "MaximilianLindholz/pelvis-siglip-siglip2-orientation",
}

def resolve_checkpoint_dir(local_name: str, model_key: str) -> Path:
    local = CHECKPOINTS_DIR / local_name
    if local.exists():
        return local
    repo_id = HF_REPO_IDS[model_key]
    from huggingface_hub import snapshot_download
    print(f"Downloading {repo_id} from the Hugging Face Hub...")
    return Path(snapshot_download(repo_id))

def load_release_checkpoint(ckpt_dir: Path, model_cls):
    config = json.loads((ckpt_dir / "config.json").read_text())
    classes = json.loads((ckpt_dir / "label_classes.json").read_text())
    model = model_cls(config["n_classes"])

    if config["model"] == "MedSigLIP":
        from transformers import SiglipVisionModel
        model.backbone = SiglipVisionModel.from_pretrained(ckpt_dir / "backbone")
    else:
        model.backbone.load_state_dict(torch.load(ckpt_dir / "backbone_state_dict.pt", map_location="cpu"))

    model.head = nn.Linear(config["embed_dim"], config["n_classes"])
    model.head.load_state_dict(torch.load(ckpt_dir / "head.pt", map_location="cpu"))
    model = model.to(device).eval()
    return model, config, classes

seq_dir = resolve_checkpoint_dir("Pelvis-SigLIP-MedSigLIP-Sequence", "MedSigLIP")
orient_dir = resolve_checkpoint_dir("Pelvis-SigLIP-SigLIP-2-Orientation", "SigLIP-2")

seq_model, seq_config, seq_classes = load_release_checkpoint(seq_dir, MedSigLIPModel)
orient_model, orient_config, orient_classes = load_release_checkpoint(orient_dir, SigLIP2Model)
print("Loaded both checkpoints.")

## 4. Run inference


In [ ]:
@torch.no_grad()
def finetuned_predict(slice_arr, model, config, classes, show_bar=True):
    x = preprocess_image_array(slice_arr, config["img_size"], config["mean"], config["std"], config["grayscale"]).to(device)
    probs = model.predict(x).squeeze(0).cpu().numpy()
    order = np.argsort(probs)[::-1]
    for i in order:
        marker = " <-- top" if i == order[0] else ""
        print(f"  {classes[i]:>10s}  p={probs[i]:.3f}{marker}")
    if show_bar:
        plt.figure(figsize=(5, 2.5))
        plt.bar(classes, probs)
        plt.ylabel("probability")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()
    return classes[order[0]]

print("[Fine-tuned MedSigLIP] Sequence type:")
ft_seq = finetuned_predict(slice_arr, seq_model, seq_config, seq_classes)

print("\n[Fine-tuned SigLIP-2] View orientation:")
ft_orient = finetuned_predict(slice_arr, orient_model, orient_config, orient_classes)

## 5. Summary


In [ ]:
print(f"{'Task':<14}{'Zero-shot':<14}{'Fine-tuned':<14}")
print(f"{'Sequence':<14}{zs_seq:<14}{ft_seq:<14}")
print(f"{'Orientation':<14}{zs_orient:<14}{ft_orient:<14}")

If the zero-shot and fine-tuned predictions disagree, trust the fine-tuned one — that's the entire point of the paper (Table 2 shows full fine-tuning reaches ≥93% accuracy on both tasks, vs. 14.5–29.4% zero-shot). If the fine-tuned prediction itself looks wrong, double check the extracted slice in section 1 — a bad slice/orientation is the most common cause of a bad prediction, not the model.